# UK Parliament (Hansard) debate transcripts

English demo dataset — a near-twin of `miles_guo` (large, noisy corpus of many short speeches), built by reusing the existing pipeline end-to-end. See `docs/plan/hansard-index.md`.

Pipeline: download → chunk → (summary) → index chunks → index titles. Titles come straight from the Hansard API (no LLM title extraction needed).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# Change to project root directory (parent of scripts folder)
current_dir = Path.cwd()
if current_dir.name == 'scripts':
    os.chdir(current_dir.parent)

In [ ]:
# Download Hansard debates from the official Parliament API.
# One top-level debate section = one document. Also writes titles.json directly
# from the API, so the LLM TitleExtractor step is skipped entirely.
# Default window (~9 Commons sitting days) is sized to ~1M tokens / ~11k chunks;
# widen/narrow date_ranges to scale. Public record under the Open Parliament Licence.
from lib.data.downloader_hansard import HansardDataDownloader

downloader = HansardDataDownloader(
    out_folder="./data/data_hansard/documents/",
    titles_path="./data/data_hansard/titles.json",
    house="Commons",
    date_ranges=[("2025-01-06", "2025-01-17")],
    sections=("Debate", "WestHall"),
    min_words=200,
    max_workers=6,
)
downloader.download_documents()

In [ ]:
# Split documents into token-based chunks.
# strip_headers=False: skip the miles-specific Chinese header/footer regexes.
# tiktoken self-calibrates chars-per-token, so English yields ~110-token chunks
# at chunk_size=500. Raise chunk_size (e.g. 1000) if the chunks read too small.
from lib.data.chunker import NaiveChunker

chunker = NaiveChunker(
    input_dir="./data/data_hansard/documents/",
    output_dir="./data/data_hansard/chunks/",
    chunk_size=500,
    chunk_overlap=100,
    strip_headers=False,
    reload=False,
)
chunker.run()

> **Title extraction is intentionally skipped.** `data/data_hansard/titles.json` is written directly by the downloader from the Hansard API (higher quality than regex extraction and free). The file already has the exact `{doc_id: title}` shape `TitleExtractor.run` would produce.

In [ ]:
# LLM summaries per debate (English). Powers the two-step title/summary search.
# Optional but recommended. lang="en" selects the English summary prompt.
from lib.data.summary_extractor import SummaryExtractor
import dotenv
dotenv.load_dotenv()

extractor = SummaryExtractor(
    input_dir="./data/data_hansard/documents/",
    output_file="./data/data_hansard/summaries.json",
    max_workers=8,
    lang="en",
    limit=None,
    skip_existing=True,
)
await extractor.run()

> **Context expansion (`contexter.py`) is skipped for v1** (like `lzj`/`lxb`). It is optional and can be added in a later iteration; the chunk indexer leaves `context` empty when no `contexts_path` is passed.

In [ ]:
# Index chunks into the `hansard` chunk index (reused unchanged).
# title_path + summaries_path attach doc_title / doc_summary to each chunk.
from lib.search.elastic_chunk_index import ElasticWriteClientChunks

elastic_chunk = ElasticWriteClientChunks(
    chunk_index_name="hansard",
    chunks_path="./data/data_hansard/chunks/",
    title_path="./data/data_hansard/titles.json",
    summaries_path="./data/data_hansard/summaries.json",
)

In [ ]:
# Clear + rebuild the chunk index. Uncomment clear_index() for a fresh build.
#elastic_chunk.clear_index()
elastic_chunk.insert_chunks(
    batch_size=1000,
    limit=None,
    skip_existing=True,
)

In [ ]:
# Index titles + summaries into the `hansard_titles` title index (reused unchanged).
from lib.search.elastic_title_index import ElasticWriteClientTitles

elastic_title = ElasticWriteClientTitles(
    title_index_name="hansard_titles",
    title_path="./data/data_hansard/titles.json",
    summaries_path="./data/data_hansard/summaries.json",
)

In [ ]:
# Insert all titles/summaries into the title index for two-step retrieval.
#elastic_title.clear_index()
elastic_title.insert_titles(
    limit=None,
    skip_existing=True,
)